# 06 Filter and Where

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Keep only the rows you need using <code>filter</code> / <code>where</code>, combine conditions correctly, and handle the tricky cases: nulls, operator precedence, <code>isin</code>, <code>between</code>, <code>like</code> and <code>rlike</code>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Filtering early is the single cheapest performance win in Spark: less data flows into joins, aggregations and shuffles. On Delta and Parquet, filters are also <b>pushed down</b> to the scan, so whole files can be skipped. Filter bugs (wrong precedence, forgotten nulls) silently produce wrong numbers, which is why interviewers like them.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How filtering works</b><br>
<code>filter(condition)</code> returns a new DataFrame containing only rows where the condition is <b>true</b>.
<ul><li><code>where</code> is an alias of <code>filter</code>. They are identical</li><li>The condition is a boolean Column (<code>F.col("x") &gt; 5</code>) or a SQL string (<code>"x &gt; 5"</code>)</li><li>Combine with <code>&amp;</code> (and), <code>|</code> (or), <code>~</code> (not), and wrap <b>each condition in parentheses</b></li><li>Rows where the condition is <code>null</code> are <b>dropped</b>, just like rows where it is false</li><li>It is a <b>narrow</b> transformation: no shuffle</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A fraud team scans 10 million card transactions per day but only cares about rows where <code>amount &gt; 5000</code> <b>and</b> the country differs from the card's home country, <b>or</b> the merchant category is on a watch list. Getting the parentheses wrong in that one line changes which transactions get flagged.
</div>

## Syntax

```python
from pyspark.sql import functions as F

df.filter(F.col("amount") > 1000)
df.where("amount > 1000 AND country = 'IN'")            # SQL string
df.filter((F.col("amount") > 1000) & (F.col("country") == "IN"))
df.filter(F.col("country").isin("IN", "AE"))
df.filter(F.col("amount").between(100, 500))           # inclusive on both ends
df.filter(F.col("merchant").like("%AMAZON%"))
df.filter(F.col("email").rlike(r"^[\w.]+@[\w.]+$"))
df.filter(F.col("phone").isNull())
```

| Python operator | Meaning | SQL equivalent |
|---|---|---|
| `==` / `!=` | equal / not equal | `=` / `<>` |
| `&` | and | `AND` |
| <code>&#124;</code> | or | `OR` |
| `~` | not | `NOT` |
| `eqNullSafe` | equal, treating null = null as true | `<=>` |

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a transactions DataFrame, including some nulls on purpose.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
An 8-row table. Transaction 6 has a null <code>country</code>, transaction 8 has a null <code>amount</code>.
</div>

In [0]:
from pyspark.sql import functions as F

txns = spark.createDataFrame(
    [
        (1, "C001", 120.50, "IN", "AMAZON RETAIL", "2024-03-01"),
        (2, "C002", 8700.00, "AE", "GOLD SOUK", "2024-03-01"),
        (3, "C001", 45.00, "IN", "UBER TRIP", "2024-03-02"),
        (4, "C003", 15000.00, "US", "APPLE STORE", "2024-03-02"),
        (5, "C004", 300.00, "UK", "amazon marketplace", "2024-03-03"),
        (6, "C002", 999.99, None, "UNKNOWN MERCHANT", "2024-03-03"),
        (7, "C005", 5200.00, "IN", "JEWELLERY MART", "2024-03-04"),
        (8, "C003", None, "US", "REFUND DESK", "2024-03-04"),
    ],
    "txn_id INT, customer_id STRING, amount DOUBLE, country STRING, merchant STRING, txn_date STRING",
).withColumn("txn_date", F.to_date("txn_date"))
display(txns)

## 1. Simple conditions: filter, where and SQL strings

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Applies the same condition in three styles.<br><br>
<b>Why it matters</b><br>All three compile to the same plan. Use whichever reads best, but be consistent within a project.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three identical tables with transactions 2, 4 and 7 (amount > 5000).
</div>

In [0]:
txns.filter(F.col("amount") > 5000).show()
txns.where(F.col("amount") > 5000).show()
txns.filter("amount > 5000").show()

## 2. Combining conditions: `&`, `|`, `~` and parentheses

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Combines conditions with and, or and not.<br><br>
<b>Why it matters</b><br>In Python, <code>&amp;</code> and <code>|</code> bind <b>more tightly</b> than <code>&gt;</code> and <code>==</code>. Without parentheses, <code>F.col("a") &gt; 1 &amp; F.col("b") == 2</code> is parsed as <code>F.col("a") &gt; (1 &amp; F.col("b")) == 2</code>, which errors or does something unexpected.
</div>
<br>
<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Common mistake</b><br>
Using Python's <code>and</code> / <code>or</code> / <code>not</code> keywords on Columns raises <code>ValueError: Cannot convert column into bool</code>. Always use <code>&amp;</code>, <code>|</code>, <code>~</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Big transactions in India (7), then big <b>or</b> UK transactions (2, 4, 5, 7), then everything not from India (2, 4, 5, 8). Transaction 6 is missing from the last result, see section 5.
</div>

In [0]:
# AND: both conditions, each in its own parentheses
txns.filter((F.col("amount") > 5000) & (F.col("country") == "IN")).show()

# OR
txns.filter((F.col("amount") > 5000) | (F.col("country") == "UK")).show()

# NOT
txns.filter(~(F.col("country") == "IN")).show()

In [0]:
# This is what happens with Python's 'and' keyword instead of '&'.
try:
    txns.filter((F.col("amount") > 5000) and (F.col("country") == "IN")).show()
except ValueError as e:
    print("ValueError:", str(e)[:120])

## 3. isin, between, like, rlike, startswith, contains

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses the most common helper methods on columns.<br><br>
<b>Why it matters</b><br>They are shorter and clearer than long chains of <code>|</code> conditions. <code>like</code> is case sensitive, <code>ilike</code> is not, and <code>rlike</code> takes a regular expression.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Five results: transactions in IN or AE; amounts 100 to 1000 inclusive; merchants matching <code>%AMAZON%</code> (only 1, because <code>like</code> is case sensitive); case-insensitive Amazon match (1 and 5); and merchants starting with <code>J</code> or <code>G</code>.
</div>

In [0]:
txns.filter(F.col("country").isin("IN", "AE")).show()

# between is INCLUSIVE on both ends: 100 <= amount <= 1000
txns.filter(F.col("amount").between(100, 1000)).show()

# like uses SQL wildcards: % = any characters, _ = one character
txns.filter(F.col("merchant").like("%AMAZON%")).show()

# ilike ignores case. lower() + like does the same thing on any Spark version.
txns.filter(F.col("merchant").ilike("%amazon%")).show()

# rlike takes a regular expression
txns.filter(F.col("merchant").rlike("^(J|G)")).show()

# startswith / endswith / contains are simple and readable
print(txns.filter(F.col("merchant").contains("STORE")).count(), "merchant(s) contain STORE")

## 4. Filtering on dates and building conditions in Python

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Filters a date range, then builds a condition dynamically from a Python dictionary.<br><br>
<b>Why it matters</b><br>Incremental pipelines filter on dates every run. Building conditions from config is a common pattern in reusable code.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Transactions from 2 to 3 March (3, 4, 5, 6), then the rows matching <code>country = IN</code> and <code>customer_id = C001</code> (1 and 3).
</div>

In [0]:
txns.filter(F.col("txn_date").between("2024-03-02", "2024-03-03")).show()

# Build an AND of equality conditions from a dict.
from functools import reduce

rules = {"country": "IN", "customer_id": "C001"}
condition = reduce(lambda a, b: a & b, [F.col(k) == v for k, v in rules.items()])
print(condition)          # the combined Column expression
txns.filter(condition).show()

## 5. Edge case: nulls in filters (three-valued logic)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows that comparisons with null produce null, and that <code>filter</code> drops rows where the condition is null.<br><br>
<b>Why it matters</b><br>This is the most common silent filter bug. <code>country != 'IN'</code> does <b>not</b> return rows where country is null, because <code>null != 'IN'</code> is null (unknown), not true.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>!=</code> filter returns 4 rows and misses transaction 6. Adding <code>isNull()</code> returns 5 rows. <code>eqNullSafe</code> treats null as a comparable value.
</div>

In [0]:
# The comparison result itself is null for row 6.
txns.select("txn_id", "country", (F.col("country") != "IN").alias("not_india")).show()

print("country != 'IN':", txns.filter(F.col("country") != "IN").count())
print("... OR country IS NULL:", txns.filter((F.col("country") != "IN") | F.col("country").isNull()).count())

# Null-safe equality: null <=> null is true, null <=> 'IN' is false (never null).
txns.filter(~F.col("country").eqNullSafe("IN")).select("txn_id", "country").show()

# Finding the nulls themselves
txns.filter(F.col("amount").isNull()).show()

## 6. Filter early: predicate pushdown

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a small Parquet dataset and shows that the filter appears as <code>PushedFilters</code> in the scan.<br><br>
<b>Why it matters</b><br>When the filter is pushed into the scan, Spark uses file and row-group statistics (min/max per column) to skip data it doesn't need to read. With Delta on Databricks this becomes <b>data skipping</b>, which you will study in the Delta Lake folder.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
On serverless, writing to a temporary path is done through a <b>Unity Catalog volume</b> (DBFS root is disabled). To keep this lesson simple, the cell uses a managed table instead, which works on Free Edition with no setup.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A physical plan with a scan node whose details include a filter on <code>amount</code>, such as <code>PushedFilters: [IsNotNull(amount), GreaterThan(amount,5000.0)]</code> or a <code>DataFilters</code> / data-skipping entry on Databricks.
</div>

In [0]:
# Save as a managed table in your current catalog and schema (workspace.default on Free Edition).
txns.write.mode("overwrite").saveAsTable("lesson06_txns")

# Read it back with a filter and look at the scan in the plan.
spark.table("lesson06_txns").filter(F.col("amount") > 5000).explain()

In [0]:
# Clean up the table so your schema stays tidy.
spark.sql("DROP TABLE IF EXISTS lesson06_txns")

## Under the hood

```
txns.filter((col("amount") > 5000) & (col("country") == "IN"))
        |
        v
 Logical plan:   Filter ((amount > 5000) AND (country = IN))
        |
        v
 Optimizer:      - combines consecutive filters into one
                 - adds IsNotNull(...) checks
                 - pushes the filter below projects and into the scan
        |
        v
 Physical plan:  Scan with PushedFilters -> Filter -> (rest of query)
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> `filter` adds a **Filter** node. Nothing runs until an action.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> each partition is filtered independently.

The optimizer moves filters as early as possible (**predicate pushdown**), even if you write them late in your code. It also merges consecutive `filter` calls. Check that in the cell below: two filters in the code, one `Filter` in the optimized plan.

In [0]:
(
    spark.range(1_000_000)
    .withColumn("amount", F.col("id") % 10_000)
    .filter(F.col("amount") > 5000)
    .filter(F.col("id") % 2 == 0)
).explain(True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>ValueError: Cannot convert column into bool</code></b><br>
You used <code>and</code>, <code>or</code>, <code>not</code> or a Python <code>if</code> on a Column. Use <code>&amp;</code>, <code>|</code>, <code>~</code>.<br><br>
<b>⛔ Problem: wrong rows when combining conditions</b><br>Missing parentheses. Wrap every comparison: <code>(cond1) &amp; (cond2)</code>.<br><br>
<b>⛔ Problem: rows with nulls disappear</b><br><code>!=</code>, <code>&lt;</code>, <code>&gt;</code> with null give null, and null rows are dropped. Add <code>| F.col(x).isNull()</code> or use <code>eqNullSafe</code>.<br><br>
**⛔ Problem: <code>like</code> misses rows**<br><code>like</code> is case sensitive. Use <code>ilike</code>, or <code>F.lower(col).like(...)</code>.<br><br>
**⛔ Problem: <code>between</code> includes a boundary you didn't want**<br><code>between</code> is inclusive. Use <code>(col &gt;= a) &amp; (col &lt; b)</code> for half-open ranges, which is safer for timestamps.<br><br>
<b>⛔ Problem: date filter compares strings</b><br>If the column is a string, <code>"2024-3-2" &gt; "2024-03-10"</code> compares text. Convert with <code>to_date</code> first.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between <code>filter</code> and <code>where</code>?</b><br>
None. <code>where</code> is an alias for <code>filter</code>, added for SQL users. Both accept a Column condition or a SQL string.<br><br>

<b>🎤 2. Why do you need parentheses around conditions in PySpark?</b><br>
Because Python gives <code>&amp;</code> and <code>|</code> higher precedence than comparison operators, so <code>a &gt; 1 &amp; b &lt; 2</code> is parsed wrongly. Wrapping each comparison in parentheses fixes it.<br><br>

<b>🎤 3. Why can't you use Python's <code>and</code> / <code>or</code>?</b><br>
They call <code>bool()</code> on the Column, and a Column has no single True/False value until Spark evaluates it per row, so PySpark raises a <code>ValueError</code>.<br><br>

<b>🎤 4. How does <code>filter</code> treat nulls?</b><br>
A comparison with null returns null, and <code>filter</code> keeps only rows where the condition is true, so those rows are dropped. You must handle nulls explicitly with <code>isNull</code>, <code>isNotNull</code>, <code>coalesce</code> or <code>eqNullSafe</code>.<br><br>

<b>🎤 5. What is predicate pushdown?</b><br>
The optimizer moves filters down to the data source scan. With Parquet and Delta, the scan uses file statistics to skip files or row groups that can't match, so less data is read.<br><br>

<b>🎤 6. Is <code>filter</code> a narrow or wide transformation?</b><br>
Narrow. Each partition is filtered on its own, with no data movement between executors.<br><br>

<b>🎤 7. Does the position of a filter in your code matter for performance?</b><br>
Usually not much, because the optimizer pushes filters early. But it can't push a filter past a Python UDF or some complex operations, so filtering early in code is still a good habit.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A column <code>status</code> contains <code>'A'</code>, <code>'B'</code> and NULL. Which rows does <code>df.filter("status &lt;&gt; 'A'")</code> return?</b><br>
A. Rows with <code>'B'</code> and NULL<br>
B. Only rows with <code>'B'</code><br>
C. Only rows with NULL<br>
D. All rows<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>NULL &lt;&gt; 'A'</code> evaluates to NULL, which is treated as false by a filter.</details><br><br>

<b>Q2. Which PySpark expression correctly keeps rows where <code>qty &gt; 10</code> and <code>region</code> is <code>'EU'</code>?</b><br>
A. <code>df.filter(col("qty") &gt; 10 &amp; col("region") == "EU")</code><br>
B. <code>df.filter((col("qty") &gt; 10) and (col("region") == "EU"))</code><br>
C. <code>df.filter((col("qty") &gt; 10) &amp; (col("region") == "EU"))</code><br>
D. <code>df.filter(col("qty") &gt; 10, col("region") == "EU")</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> Parentheses plus <code>&amp;</code>. A has a precedence bug, B uses Python <code>and</code>, D is not valid syntax.</details><br><br>

<b>Q3. Which SQL operator compares two values and treats two NULLs as equal?</b><br>
A. <code>=</code><br>
B. <code>==</code><br>
C. <code>&lt;=&gt;</code><br>
D. <code>IS</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> <code>&lt;=&gt;</code> is the null-safe equality operator (<code>eqNullSafe</code> in PySpark).</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>txns</code>:
<ol><li>Return transactions over 1000 from the UK or the US</li><li>Return transactions <b>not</b> from India, <b>including</b> the one with an unknown country</li><li>Return transactions whose merchant contains "amazon" in any case</li><li>Return transactions from 1 to 2 March 2024 with an amount between 40 and 200</li><li>Count rows with a missing amount or missing country</li><li>Write task 1 as a SQL string with <code>where</code></li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Amount over 1000 AND country in UK/US
txns.filter((F.col("amount") > 1000) & F.col("country").isin("UK", "US")).show()

# 2. Not India, keeping the null country
txns.filter((F.col("country") != "IN") | F.col("country").isNull()).show()

# 3. Case-insensitive match
txns.filter(F.lower("merchant").contains("amazon")).show()

# 4. Date range and amount range
txns.filter(
    F.col("txn_date").between("2024-03-01", "2024-03-02") & F.col("amount").between(40, 200)
).show()

# 5. Missing values
print(txns.filter(F.col("amount").isNull() | F.col("country").isNull()).count())

# 6. SQL string version of task 1
txns.where("amount > 1000 AND country IN ('UK', 'US')").show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>filter</code> and <code>where</code> are the same narrow, lazy transformation</li><li>Use <code>&amp;</code>, <code>|</code>, <code>~</code> and wrap each condition in parentheses</li><li><code>isin</code>, <code>between</code> (inclusive), <code>like</code> (case sensitive), <code>ilike</code>, <code>rlike</code>, <code>contains</code>, <code>startswith</code></li><li>Comparisons with null return null, and those rows are dropped. Handle nulls explicitly</li><li><code>eqNullSafe</code> / <code>&lt;=&gt;</code> treats null as a value</li><li>The optimizer merges filters and pushes them into the scan (predicate pushdown)</li><li>Filter early to reduce data before joins and aggregations</li></ul>
</div>

| Task | Code |
|---|---|
| Compare | `df.filter(F.col("x") > 5)` |
| And / or / not | `(c1) & (c2)`, <code>(c1) &#124; (c2)</code>, `~c1` |
| In list | `F.col("x").isin("a", "b")` |
| Range | `F.col("x").between(1, 10)` |
| Pattern | `like("%a%")`, `ilike(...)`, `rlike("^a")` |
| Nulls | `isNull()`, `isNotNull()`, `eqNullSafe(v)` |
| SQL | `df.where("x > 5 AND y = 'a'")` |

## Git commit

```
git add 01_spark_basics/06_filter_where.ipynb
git commit -m "add 06 filter where notebook"
```